In [1]:
import pandas as pd
manifest = pd.read_csv(r'C:\Users\jaide\Downloads\intel_robotic_welding_dataset\manifest.csv')
manifest.head()

,CATEGORY,WELD_TYPE,THICKNESS_MM,STEEL_TYPE,SAMPLES,CURRENT_A,VOLTAGE_V,GAS_BAR,ROBOT_SPEED_CPM,DIRECTORY,SUBDIRS,SPLIT
0,Good,FILLET,7,FE410,19,165,21.0,4.5,25,2_good_weld_2_02-09-23_Fe410,2_good_weld_2_02-09-23_Fe410/04-01-23-0024-00,TRAIN
1,Good,NON_FILLET,7,FE410,20,160,22.0,4.0,30,good_weld_28_12-13-22_butt,good_weld_28_12-13-22_butt/12-13-22-0423-00,TRAIN
2,Good,NON_FILLET,7,FE410,20,150,19.0,4.0,30,good_weld_24_12-13-22_butt,good_weld_24_12-13-22_butt/12-13-22-0355-00,TRAIN
3,Good,NON_FILLET,7,FE410,20,170,22.0,4.0,30,good_weld_4_09-26-22_plane_plate,good_weld_4_09-26-22_plane_plate/09-26-22-0032-00,TRAIN
4,Good,NON_FILLET,7,FE410,20,165,21.0,4.0,30,good_weld_29_12-13-22_butt,good_weld_29_12-13-22_butt/12-13-22-0452-00,TRAIN


In [2]:
manifest.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4040 entries, 0 to 4039
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   CATEGORY         4040 non-null   object 
 1   WELD_TYPE        4040 non-null   object 
 2   THICKNESS_MM     4040 non-null   int64  
 3   STEEL_TYPE       4040 non-null   object 
 4   SAMPLES          4040 non-null   int64  
 5   CURRENT_A        4040 non-null   int64  
 6   VOLTAGE_V        4040 non-null   float64
 7   GAS_BAR          4040 non-null   float64
 8   ROBOT_SPEED_CPM  4040 non-null   int64  
 9   DIRECTORY        4040 non-null   object 
 10  SUBDIRS          4040 non-null   object 
 11  SPLIT            4040 non-null   object 
dtypes: float64(2), int64(4), object(6)
memory usage: 378.9+ KB


In [3]:
manifest['CATEGORY'].value_counts()

Good                                819
Excessive_Penetration               480
Porosity_w_Excessive_Penetration    480
Porosity                            340
Warping                             320
Burnthrough                         320
Lack_of_Fusion                      320
Spatter                             320
Crater_Cracks                       161
Excessive_Convexity                 160
Undercut                            160
Overlap                             160
Name: CATEGORY, dtype: int64

In [4]:
import os

BASE_PATH = r'C:\Users\jaide\Downloads\intel_robotic_welding_dataset'

In [5]:
def group_defects(x):
    if x == 'Good':
        return '1'
    else:
        return '0'
    
manifest['CATEGORY'] = manifest['CATEGORY'].apply(group_defects)

manifest['CATEGORY'].value_counts()

0    3221
1     819
Name: CATEGORY, dtype: int64

In [6]:
manifest.drop(columns=['SPLIT'],inplace=True)

In [7]:
manifest = pd.get_dummies(manifest, columns=['STEEL_TYPE', 'WELD_TYPE'])

manifest.head()

,CATEGORY,THICKNESS_MM,SAMPLES,CURRENT_A,VOLTAGE_V,GAS_BAR,ROBOT_SPEED_CPM,DIRECTORY,SUBDIRS,STEEL_TYPE_BSK46,STEEL_TYPE_FE410,WELD_TYPE_FILLET,WELD_TYPE_NON_FILLET
0,1,7,19,165,21.0,4.5,25,2_good_weld_2_02-09-23_Fe410,2_good_weld_2_02-09-23_Fe410/04-01-23-0024-00,0,1,1,0
1,1,7,20,160,22.0,4.0,30,good_weld_28_12-13-22_butt,good_weld_28_12-13-22_butt/12-13-22-0423-00,0,1,0,1
2,1,7,20,150,19.0,4.0,30,good_weld_24_12-13-22_butt,good_weld_24_12-13-22_butt/12-13-22-0355-00,0,1,0,1
3,1,7,20,170,22.0,4.0,30,good_weld_4_09-26-22_plane_plate,good_weld_4_09-26-22_plane_plate/09-26-22-0032-00,0,1,0,1
4,1,7,20,165,21.0,4.0,30,good_weld_29_12-13-22_butt,good_weld_29_12-13-22_butt/12-13-22-0452-00,0,1,0,1


In [8]:
def get_csv_path(row):
    filename = row['SUBDIRS'].split('/')[-1] + ".csv"
    
    return os.path.join(
        BASE_PATH,
        row['SUBDIRS'],
        filename
    )

In [9]:
path = get_csv_path(manifest.iloc[0])

print(path)
print(os.path.exists(path))

C:\Users\jaide\Downloads\intel_robotic_welding_dataset\2_good_weld_2_02-09-23_Fe410/04-01-23-0024-00\04-01-23-0024-00.csv
True


In [10]:
import numpy as np

def extract_features(df):
    features = {}
    
    drop_cols = ['Date','Time','Part No']
    df = df.drop(columns=[c for c in drop_cols if c in df.columns])
    
    df = df.fillna(method='ffill').fillna(0)
    
    df = df.loc[:, df.nunique() > 1]
    
    for col in df.columns:
        features[f'{col}_mean'] = df[col].mean()
        features[f'{col}_std'] = df[col].std()
        features[f'{col}_min'] = df[col].min()
        features[f'{col}_max'] = df[col].max()

    return features

In [11]:
final_data = []

for _, row in manifest.iterrows():
    path = get_csv_path(row)

    if os.path.exists(path):
        try:
            ts_df = pd.read_csv(path)
            ts_features = extract_features(ts_df)

            combined = row.to_dict()
            combined.update(ts_features)

            final_data.append(combined)

        except Exception as e:
            print(f"Error in {path}: {e}")

In [12]:
final_df = pd.DataFrame(final_data)
print(final_df.shape)

(4040, 37)


In [13]:
final_df.head()

,CATEGORY,THICKNESS_MM,SAMPLES,CURRENT_A,VOLTAGE_V,GAS_BAR,ROBOT_SPEED_CPM,DIRECTORY,SUBDIRS,STEEL_TYPE_BSK46,...,Primary Weld Current_min,Primary Weld Current_max,Wire Consumed_mean,Wire Consumed_std,Wire Consumed_min,Wire Consumed_max,Secondary Weld Voltage_mean,Secondary Weld Voltage_std,Secondary Weld Voltage_min,Secondary Weld Voltage_max
0,1,7,19,165,21.0,4.5,25,2_good_weld_2_02-09-23_Fe410,2_good_weld_2_02-09-23_Fe410/04-01-23-0024-00,0,...,0.0,193.30,3.677552,1.257343,0.98,5.90,12.830985,8.492224,0.0,40.23
1,1,7,20,160,22.0,4.0,30,good_weld_28_12-13-22_butt,good_weld_28_12-13-22_butt/12-13-22-0423-00,0,...,0.0,197.05,722.036826,520.132619,12.79,1452.46,13.639162,10.272898,0.0,81.91
2,1,7,20,150,19.0,4.0,30,good_weld_24_12-13-22_butt,good_weld_24_12-13-22_butt/12-13-22-0355-00,0,...,0.0,190.19,667.018795,486.358377,13.11,1355.08,14.256416,9.715918,0.0,81.90
3,1,7,20,170,22.0,4.0,30,good_weld_4_09-26-22_plane_plate,good_weld_4_09-26-22_plane_plate/09-26-22-0032-00,0,...,0.0,196.48,903.518298,664.202286,9.34,1854.43,16.749818,10.683278,0.0,40.23
4,1,7,20,165,21.0,4.0,30,good_weld_29_12-13-22_butt,good_weld_29_12-13-22_butt/12-13-22-0452-00,0,...,0.0,196.09,783.038114,577.300168,9.18,1598.69,15.231886,11.760270,0.0,81.92


In [14]:
final_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4040 entries, 0 to 4039
Data columns (total 37 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   CATEGORY                     4040 non-null   object 
 1   THICKNESS_MM                 4040 non-null   int64  
 2   SAMPLES                      4040 non-null   int64  
 3   CURRENT_A                    4040 non-null   int64  
 4   VOLTAGE_V                    4040 non-null   float64
 5   GAS_BAR                      4040 non-null   float64
 6   ROBOT_SPEED_CPM              4040 non-null   int64  
 7   DIRECTORY                    4040 non-null   object 
 8   SUBDIRS                      4040 non-null   object 
 9   STEEL_TYPE_BSK46             4040 non-null   int64  
 10  STEEL_TYPE_FE410             4040 non-null   int64  
 11  WELD_TYPE_FILLET             4040 non-null   int64  
 12  WELD_TYPE_NON_FILLET         4040 non-null   int64  
 13  Pressure_mean     

In [20]:
final_df = final_df.drop(columns=['DIRECTORY', 'SUBDIRS','SAMPLES'], errors='ignore')

In [21]:
X = final_df.drop(columns=['CATEGORY'])
y = final_df['CATEGORY']

In [22]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42,shuffle=True)

from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=100, random_state=42)

In [23]:
model.fit(X_train, y_train)

RandomForestClassifier(bootstrap=True, class_weight=None, criterion='gini',
            max_depth=None, max_features='auto', max_leaf_nodes=None,
            min_impurity_decrease=0.0, min_impurity_split=None,
            min_samples_leaf=1, min_samples_split=2,
            min_weight_fraction_leaf=0.0, n_estimators=100, n_jobs=1,
            oob_score=False, random_state=42, verbose=0, warm_start=False)

In [24]:
from sklearn.metrics import classification_report

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

             precision    recall  f1-score   support

          0       1.00      1.00      1.00       625
          1       1.00      0.99      1.00       183

avg / total       1.00      1.00      1.00       808



In [26]:
importance = pd.Series(model.feature_importances_, index=X.columns)
importance = importance.sort_values(ascending=False)

importance

Pressure_max                   0.146021
Pressure_std                   0.110772
GAS_BAR                        0.107136
Pressure_mean                  0.068582
VOLTAGE_V                      0.059747
THICKNESS_MM                   0.058233
Secondary Weld Voltage_mean    0.049561
CO2 Weld Flow_max              0.044326
CO2 Weld Flow_std              0.040448
CO2 Weld Flow_mean             0.031122
Wire Consumed_std              0.027517
Primary Weld Current_mean      0.023389
CURRENT_A                      0.022752
Pressure_min                   0.021870
Wire Consumed_max              0.019344
Feed_max                       0.018763
Primary Weld Current_std       0.018645
CO2 Weld Flow_min              0.017256
Wire Consumed_mean             0.016889
Primary Weld Current_max       0.015219
Feed_mean                      0.013771
Secondary Weld Voltage_std     0.013752
Feed_std                       0.013712
Wire Consumed_min              0.008965
ROBOT_SPEED_CPM                0.006117


In [27]:
#Checking how time series features are affecting, check for any leakage
cols = ['CURRENT_A', 'VOLTAGE_V', 'GAS_BAR', 'THICKNESS_MM']

X_simple = final_df[cols]

X_train, X_test, y_train, y_test = train_test_split(
    X_simple, y, test_size=0.2, random_state=42
)

model.fit(X_train, y_train)

from sklearn.metrics import classification_report
print(classification_report(y_test, model.predict(X_test)))

             precision    recall  f1-score   support

          0       0.97      0.98      0.98       625
          1       0.93      0.91      0.92       183

avg / total       0.96      0.96      0.96       808



The static parameters work well on their own. Thus adding time series features make it even better

In [28]:
df2 = pd.DataFrame(final_data)

In [29]:
#ensuring data from same weld doesnt appear in both training and testing sets, to prevent any data leakage
from sklearn.model_selection import GroupShuffleSplit

groups = df2['SUBDIRS']

gss = GroupShuffleSplit(test_size=0.2, random_state=42)

train_idx, test_idx = next(gss.split(X, y, groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

In [30]:
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))

             precision    recall  f1-score   support

          0       1.00      1.00      1.00       635
          1       0.99      0.99      0.99       173

avg / total       1.00      1.00      1.00       808

